# 09 — LLMs open-source via Ollama em GPU (Google Colab / Kaggle)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/luanfreitas5/sentimento-ptbr-llm/blob/main/notebooks/09_cloud_gpu_llm.ipynb)

Executa a etapa **`training_llm`** (seleção dos exemplos few-shot + validação) e a avaliação no teste
dos LLMs configurados em `configs/model_params.yaml -> llm`, usando um servidor **Ollama** instalado na
própria máquina da nuvem (a GPU acelera a inferência). O código é o do repositório
(`pipelines.training_llm.run_training_llm_stage`); não há ajuste de pesos — LLMs aqui são
classificadores *few-shot*, então o custo é de **inferência** (uma chamada por texto).

## Antes de começar

| | Colab | Kaggle |
|---|---|---|
| GPU | Runtime → **T4 GPU** (L4/A100 no Pro para modelos de 7–8B) | Settings → **GPU T4 x2** ou **P100** |
| Dados | `treino/validacao/teste.parquet` em `MyDrive/sentimento-ptbr-llm/data/gold/` | Dataset anexado com os 3 arquivos |
| Internet | já habilitada | Settings → **Internet On** (instala o Ollama e baixa o modelo) |
| Resultados | Drive (`MyDrive/sentimento-ptbr-llm/outputs/`) | Aba **Output** (`/kaggle/working/outputs/`) |

**Primeiro rode com `SMOKE_TEST = True`.** As predições são salvas por lote: se a sessão cair, reexecute
e o notebook retoma de onde parou. Modelos de 3B (`llama3.2`) cabem folgadamente na T4; modelos de
~8B (`llama3.1:8b`, quantizados em 4 bits pelo Ollama) também cabem (≈5 GB).

## 1. Plataforma e GPU

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
PLATFORM = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"
print(f"Plataforma detectada: {PLATFORM}")

import torch

if torch.cuda.is_available():
    gpu_properties = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu_properties.name} | VRAM: {gpu_properties.total_memory / 1e9:.1f} GB")
else:
    print(
        "ATENÇÃO: nenhuma GPU detectada. Colab: Runtime > Change runtime type > GPU. "
        "Kaggle: Settings > Accelerator > GPU T4 x2 / P100."
    )

## 2. Parâmetros da execução

In [ ]:
# --- Repositório -----------------------------------------------------------
REPO_URL = "https://github.com/luanfreitas5/sentimento-ptbr-llm.git"
REPO_BRANCH = "main"

# --- Onde estão os dados (treino/validacao/teste.parquet) -------------------
# Colab : pasta do Google Drive que contém treino.parquet, validacao.parquet e teste.parquet
#         (em qualquer subpasta). Caminho relativo a /content/drive/MyDrive.
# Kaggle: os arquivos devem estar num Dataset anexado ao notebook (Add Input);
#         o notebook procura em /kaggle/input/* automaticamente.
DRIVE_DATA_DIR = "sentimento-ptbr-llm/data/gold"
DRIVE_OUTPUT_DIR = "sentimento-ptbr-llm/outputs"  # Colab: onde salvar os resultados

# --- Execução ---------------------------------------------------------------
NOTEBOOK_TAG = "llm"

# chave do artefato (models/checkpoints/<chave>.llm.json) -> modelo do Ollama (`ollama pull`).
# "llama3_2" é a chave original do projeto (configs/model_params.yaml -> llm).
LLM_MODELS = {
    "llama3_2": "llama3.2",  # 3B, rápido
    # "llama3_1_8b": "llama3.1:8b",  # 8B, mais preciso e ~2-3x mais lento
    # "qwen2_5_7b": "qwen2.5:7b",
}

SMOKE_TEST = True  # True: poucos textos (valida o pipeline). False: amostras completas.
SMOKE_SAMPLES = 30
MAX_VALIDATION_SAMPLES = 500  # textos de validação enviados ao LLM (custo de inferência)
MAX_TEST_SAMPLES = (
    500  # = configs/evaluation.yaml -> evaluate.llm_max_test_samples; None = todo o teste
)
CHUNK_SIZE = 25  # predições salvas a cada lote (retomada após queda de sessão)
RANDOM_SEED = 42
FORCE_RERUN = False
TRACK_WITH_MLFLOW = False
AUTO_DISCONNECT = False  # Colab: encerra o runtime ao terminar

## 3. Repositório, dados e pastas

In [ ]:
# --- Pastas de trabalho ----------------------------------------------------
WORK_DIR = Path("/kaggle/working") if IN_KAGGLE else Path("/content") if IN_COLAB else Path.cwd()
REPO_DIR = WORK_DIR / "sentimento-ptbr-llm"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DATA_SEARCH_DIRS = [Path("/content/drive/MyDrive") / DRIVE_DATA_DIR, Path("/content/data")]
    OUTPUT_ROOT = Path("/content/drive/MyDrive") / DRIVE_OUTPUT_DIR
elif IN_KAGGLE:
    DATA_SEARCH_DIRS = [path for path in Path("/kaggle/input").glob("*") if path.is_dir()]
    OUTPUT_ROOT = Path("/kaggle/working/outputs")  # tudo em /kaggle/working vira output do notebook
else:
    DATA_SEARCH_DIRS = [REPO_DIR / "data" / "gold"]
    OUTPUT_ROOT = WORK_DIR / "outputs"

# Separa execução de teste rápido (smoke) da execução completa: evita reaproveitar
# por engano um checkpoint treinado com poucos dados.
RUN_TAG = "smoke" if SMOKE_TEST else "full"
PERSIST_DIR = OUTPUT_ROOT / NOTEBOOK_TAG / RUN_TAG
PERSIST_DIR.mkdir(parents=True, exist_ok=True)
print(f"Resultados serão salvos em: {PERSIST_DIR}")


def get_secret(name: str) -> str | None:
    """Lê um segredo (Colab: Secrets; Kaggle: Add-ons > Secrets; senão, variável de ambiente)."""
    try:
        if IN_COLAB:
            from google.colab import userdata

            return userdata.get(name)
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient

            return UserSecretsClient().get_secret(name)
    except Exception:  # segredo inexistente ou sem acesso concedido ao notebook
        return None
    return os.environ.get(name)


# --- Clona (ou atualiza) o repositório --------------------------------------
if not (REPO_DIR / ".git").exists():
    token = get_secret("GITHUB_TOKEN")  # só necessário se o repositório for privado
    clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, clone_url, str(REPO_DIR)],
        check=True,
        capture_output=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], check=True
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
print(
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "log", "-1", "--oneline"],
        check=False,
        capture_output=True,
        text=True,
    ).stdout
)

# --- Copia os parquets de dados para data/gold do repositório ---------------
GOLD_DIR = REPO_DIR / "data" / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)
for file_name in ("treino.parquet", "validacao.parquet", "teste.parquet"):
    found = next(
        (match for base in DATA_SEARCH_DIRS if base.exists() for match in base.rglob(file_name)),
        None,
    )
    if found is None:
        raise FileNotFoundError(
            f"'{file_name}' não encontrado em {[str(p) for p in DATA_SEARCH_DIRS]}. "
            "Envie os parquets de data/gold/ para o Drive (Colab) ou crie um Dataset (Kaggle)."
        )
    if found.resolve() != (GOLD_DIR / file_name).resolve():
        shutil.copy2(found, GOLD_DIR / file_name)
    print(f"OK  {file_name}  <-  {found}")

sys.path.insert(0, str(REPO_DIR / "src"))  # o projeto importa módulos a partir de src/
os.chdir(REPO_DIR)

## 4. Dependências Python

In [ ]:
import subprocess
import sys

PACKAGES = [
    "pydantic",
    "pydantic-settings",
    "python-dotenv",
    "pyyaml",
    "joblib",
    "pandas",
    "pandera[polars]",
    "polars",
    "pyarrow",
    "scikit-learn",
    "scipy",
    "statsmodels",
    "xgboost",
    "mlflow",
    "rich",
    "matplotlib",
    "seaborn",
    "ollama",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PACKAGES], check=True)

## 5. Servidor Ollama
Instala o Ollama, inicia o servidor em segundo plano e baixa os modelos. A GPU é detectada
automaticamente (confirme na saída de `ollama ps`, coluna PROCESSOR, ao final da seção 8).

In [ ]:
import time
import urllib.request

OLLAMA_URL = "http://localhost:11434"


def is_ollama_up() -> bool:
    """Verifica se o servidor Ollama responde."""
    try:
        urllib.request.urlopen(OLLAMA_URL, timeout=2)
    except OSError:
        return False
    return True


if shutil.which("ollama") is None:
    subprocess.run(
        "apt-get -qq update && apt-get -qq install -y zstd pciutils", shell=True, check=True
    )
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

if not is_ollama_up():
    with (WORK_DIR / "ollama_server.log").open("w") as server_log:
        subprocess.Popen(["ollama", "serve"], stdout=server_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        if is_ollama_up():
            break
        time.sleep(1)
    else:
        raise RuntimeError(f"Servidor Ollama não respondeu; veja {WORK_DIR / 'ollama_server.log'}")
print("Servidor Ollama ativo.")

for ollama_name in LLM_MODELS.values():
    print(f"Baixando {ollama_name} ...")
    subprocess.run(["ollama", "pull", ollama_name], check=True)
print(subprocess.run(["ollama", "list"], check=False, capture_output=True, text=True).stdout)

## 6. Carga dos dados e hiperparâmetros

In [ ]:
import logging
from collections import Counter

from rich.logging import RichHandler

from config.constants import CONFIG_FILE_NAMES
from config.paths import CONFIGS_DIR, load_project_paths
from data.loader import load_training_example_dataset
from io_utils.yaml import read_yaml

logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    handlers=[RichHandler(rich_tracebacks=True)],
    force=True,
)

paths = load_project_paths()
paths.models_checkpoints_dir.mkdir(parents=True, exist_ok=True)


def load_split(path: Path) -> tuple[list[str], list[str]]:
    """Lê textos e rótulos de uma partição parquet."""
    corpus = load_training_example_dataset(path)
    return corpus["text"].to_list(), corpus["sentiment_label"].to_list()


X_train, y_train = load_split(paths.training_corpus_file)
X_val, y_val = load_split(paths.validation_corpus_file)
X_test, y_test = load_split(paths.test_corpus_file)

for split_name, labels in (("treino", y_train), ("validação", y_val), ("teste", y_test)):
    print(f"{split_name:>10}: {len(labels):>7} textos | {dict(Counter(labels))}")

from pipelines.training_llm import select_sample_indices

n_val = SMOKE_SAMPLES if SMOKE_TEST else MAX_VALIDATION_SAMPLES
n_test = SMOKE_SAMPLES if SMOKE_TEST else MAX_TEST_SAMPLES

# Hiperparâmetros-base do YAML (llama3_2); cada modelo extra herda e troca só o model_name.
llm_base_params = read_yaml(CONFIGS_DIR / CONFIG_FILE_NAMES["model_params"])["llm"]["llama3_2"]
model_params = {
    key: {**llm_base_params, "model_name": ollama_name} for key, ollama_name in LLM_MODELS.items()
}
for key, params in model_params.items():
    print(key, params)

test_indices = select_sample_indices(len(X_test), n_test, RANDOM_SEED)
X_test_sample = [X_test[i] for i in test_indices]
y_test_sample = [y_test[i] for i in test_indices]
print(f"validação enviada ao LLM: {n_val} | teste: {len(X_test_sample)}")

## 7. Etapa `training_llm` (exemplos few-shot + métricas na validação)

In [ ]:
from pipelines.training_llm import run_training_llm_stage

persist_checkpoints = PERSIST_DIR / "checkpoints"
persist_checkpoints.mkdir(parents=True, exist_ok=True)

training_results = run_training_llm_stage(
    X_train,
    y_train,
    X_val,
    y_val,
    model_names=tuple(LLM_MODELS),
    model_params=model_params,
    checkpoints_dir=paths.models_checkpoints_dir,
    max_validation_samples=n_val,
    random_seed=RANDOM_SEED,
    track_with_mlflow=TRACK_WITH_MLFLOW,
)
for key, result in training_results.items():
    print(f"[{key}] {result.elapsed_seconds / 60:.1f} min | validação: {result.metrics}")
    shutil.copy2(
        paths.models_checkpoints_dir / f"{key}.llm.json", persist_checkpoints / f"{key}.llm.json"
    )

## 8. Avaliação no teste, em lotes com retomada
As predições de cada lote são gravadas em disco; ao reexecutar, apenas os lotes faltantes são processados.

In [ ]:
import json

import polars as pl
from tqdm.auto import tqdm

from evaluation.evaluator import evaluate_classifier
from evaluation.reports import (
    build_evaluation_report,
    merge_evaluation_reports,
    save_evaluation_report,
)
from visualization.confusion_matrix import plot_confusion_matrix_heatmap
from visualization.theme import apply_project_theme, save_figure

apply_project_theme()
persist_figures = PERSIST_DIR / "figures"
persist_reports = PERSIST_DIR / "reports"
for folder in (persist_figures, persist_reports):
    folder.mkdir(parents=True, exist_ok=True)

evaluation_reports = []
for key, training_result in training_results.items():
    model = training_result.model
    cache_file = persist_reports / f"predicoes_parciais_{key}.json"
    predictions = (
        []
        if FORCE_RERUN or not cache_file.exists()
        else json.loads(cache_file.read_text(encoding="utf-8"))
    )
    if predictions:
        print(f"[{key}] retomando: {len(predictions)}/{len(X_test_sample)} predições em cache.")

    started_at = time.perf_counter()
    with tqdm(total=len(X_test_sample), initial=len(predictions), desc=f"{key}") as progress:
        while len(predictions) < len(X_test_sample):
            chunk = X_test_sample[len(predictions) : len(predictions) + CHUNK_SIZE]
            predictions.extend(model.predict(chunk).tolist())
            cache_file.write_text(json.dumps(predictions, ensure_ascii=False), encoding="utf-8")
            progress.update(len(chunk))
    elapsed = time.perf_counter() - started_at

    # predict_proba reexecutaria o LLM (dobra o custo); a confiança do LLM não é calibrada.
    result = evaluate_classifier(y_test_sample, predictions)
    print(f"[{key}] TESTE ({len(predictions)} textos): {result.point_metrics}")
    print(f"[{key}] IC95% F1-macro: {result.confidence_intervals.get('f1_macro')}")

    figure = plot_confusion_matrix_heatmap(
        result.confusion_matrix, title=f"Matriz de Confusão — {key}"
    )
    save_figure(figure, f"matriz_confusao_{key}", directory=persist_figures)
    evaluation_reports.append(build_evaluation_report(key, result))
    pl.DataFrame(
        {"text": X_test_sample, "y_true": y_test_sample, "y_pred": predictions}
    ).write_parquet(persist_reports / f"predicoes_{key}.parquet")

report = merge_evaluation_reports(evaluation_reports)
save_evaluation_report(report, persist_reports / "avaliacao_llm.csv")
ollama_ps = subprocess.run(["ollama", "ps"], check=False, capture_output=True, text=True)
print(ollama_ps.stdout)  # PROCESSOR deve indicar GPU
display(report)

## 9. Inspeção qualitativa
Lista os erros de cada modelo para a análise de erros (sarcasmo, negação, ironia etc.).

In [ ]:
for key in LLM_MODELS:
    predictions_frame = pl.read_parquet(persist_reports / f"predicoes_{key}.parquet")
    errors = predictions_frame.filter(pl.col("y_true") != pl.col("y_pred"))
    print(f"\n=== {key}: {errors.height} erros em {predictions_frame.height} textos ===")
    with pl.Config(fmt_str_lengths=140, tbl_rows=10):
        print(errors.head(10))

## 10. Salvamento

In [ ]:
archive_path = shutil.make_archive(
    str(WORK_DIR / f"resultados_{NOTEBOOK_TAG}_{RUN_TAG}"), "zip", PERSIST_DIR
)
print(f"Pacote com especificações, figuras e métricas: {archive_path}")
print("Colab : os arquivos já estão no Drive. Kaggle: use Save Version e baixe em Output.")

if AUTO_DISCONNECT and IN_COLAB:
    from google.colab import runtime

    runtime.unassign()

## Próximos passos (no seu computador)

1. Copie `checkpoints/*.llm.json` para `models/checkpoints/` e rode `make evaluate` para a avaliação
   oficial (o estágio reconstrói o classificador a partir dessa especificação e compara com os demais modelos).
2. As métricas da seção 8 são exploratórias e usam uma amostra (`MAX_TEST_SAMPLES`); com `SMOKE_TEST = True`
   **não** devem ser reportadas.
3. Para o fine-tuning de Transformers, use `08_cloud_gpu_transformer.ipynb`.